# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/humaisali/FlyRank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

**Clustering (unsupervised).**

ML-02's question was *"what performance archetypes exist across the content inventory, and
which action fits each one?"* — that's a "what kinds of items exist?" question, and the framing
skill's own mapping table sends that straight to **clustering**: no predicted target, evaluated
by cluster separation plus a human sense-check, not by ROC-AUC or precision@K.

Two checks (below) confirm this isn't secretly a classification problem in disguise:
- there's no pre-built "archetype" or "cluster" column anywhere in the starter data to predict
  — I'm not relabeling an existing rule, I'm discovering groups from scratch;
- the starter pipeline's own target, `is_declining_label` (built from `trend_direction`), is a
  *different, narrower* task — one binary flag about one axis (traffic direction). My lane asks
  whether the whole inventory resolves into a small number of recurring, multi-axis *types*, of
  which "declining" might describe only one.

Per the lane guide (section 8, Lane 3): this is metric/structural clustering — content_type,
word_count, position, CTR, engagement, and freshness numbers and buckets — never semantic
clustering, since there's no article text in this dataset.


In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/humaisali/FlyRank-ML-Internship-Starter-Repo"
REPO_DIR = "FlyRank-ML-Internship-Starter-Repo"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

# Is there already a pre-built target for this? If so, this would be classification, not clustering.
prebuilt = [c for c in df.columns if any(k in c.lower() for k in ("archetype", "cluster", "label"))]
print("Pre-built archetype/cluster/label columns in the starter data:", prebuilt or "none")

print("\nis_declining_label is not a raw column here — it's derived downstream from trend_direction,")
print("and it's a different (narrower, binary) task from mine:")
print(df["trend_direction"].value_counts())


Pre-built archetype/cluster/label columns in the starter data: none

is_declining_label is not a raw column here — it's derived downstream from trend_direction,
and it's a different (narrower, binary) task from mine:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 2. Target or proxy

**No target — clustering doesn't have one.** Instead of a label, I have to define, up front, the
feature set the grouping runs on — and just as importantly, what stays out of it.

- **In:** demand/exposure (impressions, clicks, sessions), structural facts (content_type,
  word_count, age), current-state buckets (position_tier, freshness_tier, impression_tier), and
  engagement/CTR rates — everything observable before any decision was made about the page.
- **Out — the label trap:** `trend_direction` and `trend_pct` are exactly what the starter's
  *own* supervised target is derived from. Feeding them into the clustering would let one page's
  decline/recovery direction quietly dominate the grouping, and any later "do my clusters line
  up with declining pages?" check would become circular — I'd be asking whether a group I built
  partly from decline direction correlates with decline direction.
- **Out — pseudonyms:** `content_id` and `client_id` are for joining and client-holdout splits
  only, never features.

Once real clusters exist (a later week), `trend_direction`, CTR, and engagement_rate become fair
game again — as an *external*, after-the-fact sense-check on what I found, never as inputs to
finding it.


In [2]:
exclude_ids        = {"content_id", "client_id"}          # pseudonyms: joins / client-holdout splits only
exclude_label_trap = {"trend_direction", "trend_pct"}      # what the starter's own supervised target is built from

candidate_features = [c for c in df.columns if c not in exclude_ids | exclude_label_trap]

print(f"Total columns in the starter slice:        {len(df.columns)}")
print(f"Excluded as pseudonym IDs:                  {sorted(exclude_ids)}")
print(f"Excluded by the label trap:                 {sorted(exclude_label_trap)}")
print(f"Remaining candidate feature columns:        {len(candidate_features)}")
print(candidate_features)


Total columns in the starter slice:        44
Excluded as pseudonym IDs:                  ['client_id', 'content_id']
Excluded by the label trap:                 ['trend_direction', 'trend_pct']
Remaining candidate feature columns:        40
['search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier']


## 3. Success metric

**Silhouette score, backed by a human sense-check on cluster profiles.**

There's no ground truth to score clusters against, so silhouette is the right first number: for
each page, does it sit closer to the other pages in its own cluster than to the nearest other
cluster, on a fixed -1-to-1 scale? That's defensible on its own terms without needing a label.

But a mathematically separated cluster isn't automatically a *useful* one. So "good" here is
two-part: (1) a silhouette score clearly above what naive baselines produce (e.g. random k-way
splits), and (2) for every cluster, its median impressions/CTR/engagement/freshness/position
tells a believable, actionable story a reviewer could map to protect / improve / rewrite / merge
/ prune / monitor (lane guide, section 8). A cluster that scores well but can't be described in
one sentence doesn't pass.

One thing has to be true before silhouette means anything at all: the numeric features need to
be scaled first. The columns I'd cluster on live on wildly different scales (see below) —
unscaled, a distance-based metric like silhouette would just be measuring "which pages have the
most impressions," not archetype shape.


In [3]:
from sklearn.metrics import silhouette_score  # confirms it's available; not run yet — no clustering happens in this framing notebook

scale_check = ["impressions_90d", "sessions_90d", "word_count", "ctr", "engagement_rate", "content_age_days"]
print(df[scale_check].describe().loc[["mean", "std", "min", "max"]].round(2))
print()
print("-> impressions_90d and word_count live in the thousands; ctr and engagement_rate live under ~100.")
print("   Unscaled, distance (and therefore silhouette) would be dominated by whichever column has the")
print("   biggest raw numbers — scaling every numeric feature is a precondition, not a nice-to-have.")


      impressions_90d  sessions_90d  word_count     ctr  engagement_rate  \
mean          5200.37         37.07     3107.76    0.51             2.53   
std          16838.02        107.07     1452.38    3.28             8.31   
min              1.00          1.00        8.00    0.00             0.00   
max         517715.00       4345.00     9546.00  100.00           100.00   

      content_age_days  
mean            256.17  
std             132.71  
min              90.00  
max             564.00  

-> impressions_90d and word_count live in the thousands; ctr and engagement_rate live under ~100.
   Unscaled, distance (and therefore silhouette) would be dominated by whichever column has the
   biggest raw numbers — scaling every numeric feature is a precondition, not a nice-to-have.


## 4. The unit of analysis, as a real dataframe

**One row = one content item.** The starter CSV is already deduplicated by `content_id` (30,000
rows, one pseudonymized page per row, tagged with a pseudonymized `client_id`). For the
clustering slice specifically, I'm restricting to **visible** pages — `impressions_90d >= 500`,
the same bar I used in ML-02 — because a page with almost no exposure would just describe
"nothing happened yet," not an archetype; keeping it in would create one giant near-empty
cluster that drowns out the groups I actually care about.


In [4]:
visible = df[df["impressions_90d"] >= 500].copy()
print(f"Full starter slice:                    {len(df):,} rows")
print(f"Visible pages (impressions_90d >= 500): {len(visible):,} rows ({len(visible)/len(df):.1%})")

preview_cols = ["content_id", "content_type", "position_tier", "impressions_90d",
                "ctr", "engagement_rate", "content_age_days", "freshness_tier", "word_count_tier"]
visible[preview_cols].head()


Full starter slice:                    30,000 rows
Visible pages (impressions_90d >= 500): 16,726 rows (55.8%)


,content_id,content_type,position_tier,impressions_90d,ctr,engagement_rate,content_age_days,freshness_tier,word_count_tier
0,content_304f48230142,keyword article,striking,3803,0.76,5.88,187,0-30,2000-3500
1,content_a1fb4e703a9e,keyword article,page_3_5,15320,0.05,0.00,445,0-30,2000-3500
2,content_9aa793d4d895,keyword article,page_3_5,12581,0.09,0.00,141,0-30,3500+
3,content_331d6c4de07b,keyword article,page_1,11751,0.49,1.28,463,0-30,NaN
4,content_d99b7a2d90ca,keyword article,page_3_5,19140,0.13,0.00,263,0-30,2000-3500


## 5. Why ML beats a fixed rule here

Pages don't fail (or succeed) on one axis at a time — they trip several structural/performance
signals at once, and a human-written rule tree can't hold that many interacting conditions
without becoming unreadable. The code below reuses ML-02's five simple flags (stale-visible,
declining-with-demand, thin-visible, page-one-decay, low-CTR-visible): a meaningful share of the
inventory matches two or more of them *simultaneously*. The lane guide lists seven possible
archetypes in total (champions, rising stars, hidden gems, stale visible, weak/no-demand,
engagement-problem, cannibalization-risk) — every one needs its own multi-column definition if
written by hand, and pages that straddle two of them would need yet more rules just to
disambiguate. Clustering lets a small number of learned groups absorb all of that interaction at
once, instead of an ever-growing if/else ladder someone has to maintain by hand every time a new
pattern shows up.


In [5]:
# Same five flags as ML-02, definitions copied exactly from docs/ml-intern-dataset-and-lane-guide.md, section 5
stale_visible          = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
declining_with_demand  = (df["trend_direction"] == "down") & (df["impressions_90d"] >= 100)
thin_visible           = (df["word_count"] > 0) & (df["word_count"] < 1200) & (df["impressions_90d"] >= 250)
page_one_decay         = (df["avg_position"] > 0) & (df["avg_position"] <= 10) & (df["content_age_days"] >= 180)
low_ctr_visible        = (df["impressions_90d"] >= 500) & (df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["ctr"] < 0.5)

flag_count = (stale_visible.astype(int) + declining_with_demand.astype(int) + thin_visible.astype(int)
              + page_one_decay.astype(int) + low_ctr_visible.astype(int))

print(f"Pages matching 2+ flags at once: {(flag_count >= 2).sum():,} ({(flag_count >= 2).mean()*100:.1f}%)")
print(f"Pages matching 3+ flags at once: {(flag_count >= 3).sum():,} ({(flag_count >= 3).mean()*100:.1f}%)")
print(f"Pages matching 0 flags:          {(flag_count == 0).sum():,} ({(flag_count == 0).mean()*100:.1f}%)")
print()
print("Lane guide lists 7 possible archetypes for this lane — each of the 5 flags above covers")
print("at most one axis, so overlapping pages already need 2+ rules to describe, and a full")
print("7-archetype rulebook would need many more before touching every interacting case.")


Pages matching 2+ flags at once: 8,438 (28.1%)
Pages matching 3+ flags at once: 1,991 (6.6%)
Pages matching 0 flags:          10,350 (34.5%)

Lane guide lists 7 possible archetypes for this lane — each of the 5 flags above covers
at most one axis, so overlapping pages already need 2+ rules to describe, and a full
7-archetype rulebook would need many more before touching every interacting case.


## Self-check

Before you submit, confirm each line honestly:

- [Done] Every section above is filled — markdown thinking AND the code that backs it
- [Done] The notebook runs top to bottom with no errors (Runtime → Run all)
- [Done] No client names, URLs, or private queries anywhere
- [Done] My claims use careful words: observed, measured, directional, decision-support
- [Done] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
